# Concept Notes — Proximal Gradient, Subgradient & Acceleration
**MIT 6.7220 | Nikhilesh Belulkar**

---

## 1. The Problem Structure

We want to solve:
$$\min_x F(x) = \underbrace{\frac{1}{2}\|Ax - b\|_2^2}_{f(x)\ \text{smooth}} + \underbrace{\lambda\|x\|_1}_{r(x)\ \text{non-smooth}}$$

The key insight is to **split** $F$ into two parts and treat each differently:

| Part | Function | Property | How we handle it |
|------|----------|----------|------------------|
| $f(x)$ | $\frac{1}{2}\|Ax-b\|_2^2$ | Smooth, differentiable | Gradient step |
| $r(x)$ | $\lambda\|x\|_1$ | Non-smooth | Proximal operator |

## 2. Why Can't We Just Use Gradient Descent?

Gradient descent requires $F$ to be differentiable everywhere. But $\|x\|_1 = \sum_i |x_i|$ is **not differentiable at $x_i = 0$** — it has a kink.

So we need smarter methods that can handle the non-smooth $\ell_1$ term.

## 3. The Proximal Operator

The proximal operator of a function $r$ with step size $h$ is defined as:
$$\text{prox}_{hr}(v) = \arg\min_y \left\{ r(y) + \frac{1}{2h}\|y - v\|_2^2 \right\}$$

**Intuition:** Find the point $y$ that:
- has small $r(y)$ (minimizes the non-smooth part), and
- stays close to $v$ (the $\frac{1}{2h}\|y-v\|^2$ term is a proximity penalty)

**Important:** The proximal operator gives the **exact** minimizer of this subproblem — it does NOT approximate $r$.

### For $r(x) = \lambda\|x\|_1$: Soft Thresholding

$$\text{prox}_{h\lambda\|\cdot\|_1}(v)_i = \mathcal{S}_{h\lambda}(v_i) = \begin{cases} v_i - h\lambda & \text{if } v_i > h\lambda \\ 0 & \text{if } |v_i| \leq h\lambda \\ v_i + h\lambda & \text{if } v_i < -h\lambda \end{cases}$$

This shrinks small values to zero and shifts large values toward zero — promoting sparsity.

## 4. Proximal Gradient Method

### Step 1: Where does $z_t$ come from?

Start with the subproblem we want to solve at each iteration. We linearize $f$ around $x_t$ and add a quadratic trust-region:
$$\min_y \left\{ \underbrace{\langle \nabla f(x_t),\, y - x_t \rangle}_{\text{linearized } f} + \underbrace{\frac{1}{2h}\|y - x_t\|^2}_{\text{trust region}} + r(y) \right\}$$

Focus just on the first two terms (ignore $r(y)$ for now). Let $g = \nabla f(x_t)$ and $c = x_t$ to keep it clean:
$$\langle g,\, y - c \rangle + \frac{1}{2h}\|y - c\|^2$$

**Complete the square** — pull the $\frac{1}{2h}$ out and write everything as a single squared norm:
$$= \frac{1}{2h}\left[ \|y - c\|^2 + 2h\langle g,\, y - c \rangle \right]$$
$$= \frac{1}{2h}\left[ \|y - c\|^2 + 2h\langle g,\, y - c \rangle + h^2\|g\|^2 \right] - \frac{h}{2}\|g\|^2$$
$$= \frac{1}{2h}\|y - (c - hg)\|^2 - \frac{h}{2}\|g\|^2$$

The last term $-\frac{h}{2}\|g\|^2$ is a constant (no $y$ in it), so it does not affect the minimizer. Define:
$$z_t := c - hg = x_t - h\nabla f(x_t)$$

This is just the **standard gradient step** on $f$. The full subproblem simplifies to:
$$x_{t+1} = \arg\min_y \left\{ \frac{1}{2h}\|y - z_t\|^2 + r(y) \right\} = \text{prox}_{hr}(z_t)$$

**So $z_t$ is not mysterious — it is just the gradient descent step on the smooth part $f$. The completing-the-square algebra shows that taking a gradient step on $f$ and then solving the proximal subproblem are equivalent.**

---

### Step 2: Where does the optimality condition come from?

We need to solve:
$$\min_y\; \phi(y) := r(y) + \frac{1}{2h}\|y - z_t\|^2$$

For any convex function $\phi$, a point $y^*$ is a minimizer **if and only if** zero is in its subdifferential:
$$y^* \text{ minimizes } \phi \iff 0 \in \partial \phi(y^*)$$

This is the convex analogue of "set the derivative to zero." Now compute $\partial \phi$:
$$\partial \phi(y) = \partial r(y) + \nabla\!\left[\frac{1}{2h}\|y - z_t\|^2\right]$$

The second term is smooth so we can just take its regular gradient:
$$\nabla\!\left[\frac{1}{2h}\|y - z_t\|^2\right] = \frac{1}{h}(y - z_t)$$

So the optimality condition $0 \in \partial\phi(y^*)$ becomes:
$$0 \in \partial r(y^*) + \frac{1}{h}(y^* - z_t)$$

**For $r = \lambda\|x\|_1$:** since $\|x\|_1 = \sum_i |x_i|$ is separable, the subdifferential splits component-wise:
$$0 \in \lambda\,\partial|y_i^*| + \frac{1}{h}(y_i^* - (z_t)_i) \qquad \text{for each } i$$

Each of these is a scalar problem. Solving case-by-case (exactly as in Q1) gives the **soft-thresholding** solution:
$$y_i^* = \mathcal{S}_{h\lambda}((z_t)_i)$$

**Summary of what's happening:**

$$\underbrace{z_t = x_t - h\nabla f(x_t)}_{\text{gradient step: "where would we go ignoring } r\text{?"}} \xrightarrow{\text{prox}} \underbrace{x_{t+1} = \mathcal{S}_{h\lambda}(z_t)}_{\text{pull } z_t \text{ back toward sparsity}}$$

The gradient step moves in the direction that reduces $f$. The proximal step then enforces the $\ell_1$ penalty by shrinking small components to zero — exactly solving the non-smooth part rather than approximating it.

## 5. Subgradient Method

A subgradient $g \in \partial F(x)$ generalizes the gradient to non-smooth functions. For our $F$:
$$g_t = \underbrace{A^\top(Ax_t - b)}_{\nabla f(x_t)} + \underbrace{\lambda\,\text{sign}(x_t)}_{\in\, \lambda\,\partial\|x_t\|_1}$$

The subgradient iteration is:
$$x_{t+1} = x_t - h_t\, g_t$$

### Polyak Step Size
Instead of a fixed $h$, Polyak's rule adapts the step size:
$$h_t = \frac{F(x_t) - F^*}{\|g_t\|^2}$$

### Key difference from proximal gradient:

| | Proximal Gradient | Subgradient |
|---|---|---|
| Handles $r$ | Exactly (via prox) | Approximately (via subgradient) |
| Descent guarantee | Yes, every step | No — can increase |
| Convergence rate | $O(1/t)$ | $O(1/\sqrt{t})$ |
| Step size | Fixed $h = 1/L_f$ | Adaptive (Polyak) |

## 6. Nesterov Acceleration (FISTA)

### Why accelerate?

Proximal gradient converges at $O(1/t)$. Nesterov's trick improves this to $O(1/t^2)$ — **for free**, with almost no extra computation.

### The idea: look ahead before you step

Instead of taking the gradient step from $x_t$, maintain a **momentum point** $y_t$ (an extrapolation ahead of $x_t$) and step from there:

$$x_{t+1} = \text{prox}_{hr}\!\left(y_t - h\nabla f(y_t)\right)$$

$$\alpha_{t+1} = \frac{1 + \sqrt{1 + 4\alpha_t^2}}{2}$$

$$y_{t+1} = x_{t+1} + \frac{\alpha_t - 1}{\alpha_{t+1}}(x_{t+1} - x_t)$$

The $y_{t+1}$ update extrapolates in the direction of past progress — like "overshooting" to pre-correct for the zig-zagging that normal gradient descent does.

### Convergence rates summary:

| Method | Rate | 
|--------|------|
| Subgradient | $O(1/\sqrt{t})$ |
| Proximal Gradient | $O(1/t)$ |
| FISTA (Accelerated) | $O(1/t^2)$ ← optimal for first-order methods |

## 7. Lipschitz Constant and Step Size

For the step size $h = 1/L_f$ to guarantee convergence, $L_f$ must be the **Lipschitz constant of $\nabla f$** — i.e., how fast the gradient can change:
$$\|\nabla f(x) - \nabla f(y)\|_2 \leq L_f \|x - y\|_2 \quad \forall\, x, y$$

For $f(x) = \frac{1}{2}\|Ax-b\|_2^2$:
- Hessian: $\nabla^2 f(x) = A^\top A$ (constant)
- $L_f = \lambda_{\max}(A^\top A)$ — the largest eigenvalue

**Why?** The Hessian tells you the curvature of $f$. The largest eigenvalue is the worst-case curvature. The step size $h = 1/L_f$ ensures you don't overshoot.